# Python 디자인 패턴 · 정렬 기준과 출력 방식 교체하기

같은 데이터를 다른 기준으로 정렬하거나 다른 형식으로 출력하려면 어느 줄을 바꿔야 할까요? 먼저 함수 하나를 인자로 넘겨 정렬 기준을 바꿉니다. 이어서 객체를 써서 기존 코드를 그대로 호출하면서 동작을 바꾸는 방법을 비교합니다.

함수를 직접 만들고, 리스트를 수정하면 어떤 이름에서 바뀐 값이 보이는지 설명하고, 클래스에서 인스턴스를 만들어 메서드를 호출할 수 있는 학생을 위한 선택 과정입니다. 이 동작이 낯설면 Python 문법 과정에서 먼저 연습하세요.

Python 3 커널에서 위부터 실행하세요. 입력 자료는 모든 예제에 들어 있습니다. 각 과제 셀에 코드를 작성한 뒤 자기 결과를 먼저 기록하고 바로 아래 답안을 펼치세요. 실습지 practice.md에는 짝에게 전달할 추가 입력과 기록 칸이 있습니다.

장별 추가 과제에서 쓸 분실물 접수표입니다.

| 접수 순서 | 물건 | 구역 |
| --- | --- | --- |
| 1 | 지갑 | 동문 |
| 2 | 카드 | 서문 |
| 3 | 키 | 동문 |
| 4 | 우산 | 서문 |
| 5 | 장갑 | 동문 |

[장별 추가 과제](change-requests.md)에서는 같은 접수표로 정렬 기준과 출력 형식을 바꿉니다.

### 내 결과를 다음 노트북에서 여는 준비 셀

수업 묶음의 압축을 풀고 노트북을 여세요. 이 셀은 묶음 맨 위에 `my-course` 폴더를 만듭니다. 같은 이름의 파일에 다시 저장하면 그 단계의 결과가 갱신됩니다. 뒤의 장에서 시작한다면 이 준비 셀과 해당 장의 예제를 실행하세요. 앞 장 파일이 없을 때 `read_work`의 두 번째 인자가 출발 예시가 됩니다. 낱개로 받은 노트북은 모두 같은 폴더에 둡니다.

In [ ]:
from pathlib import Path
import json

here = Path.cwd()
# 과정 폴더에서 열거나, 묶음 맨 위의 일자별 사본을 열어도 같은 장소를 씁니다.
course_root = here.parent.parent if here.parent.name == "pre-mid" else here
work = course_root / "my-course"
work.mkdir(exist_ok=True)

def save_work(name, value):
    (work / name).write_text(json.dumps(value, ensure_ascii=False, indent=2), encoding="utf-8")

def read_work(name, example):
    path = work / name
    if path.exists():
        return json.loads(path.read_text(encoding="utf-8"))
    return example

## 1장 · 같은 목록에서 정렬 기준 바꾸기

`sorted(items, key=len)`은 각 항목의 길이를 기준으로 정렬한 새 리스트를 반환합니다. `len("우산")`은 2이므로 한 글자짜리 이름이 우산보다 먼저 나옵니다. `items`에는 처음 순서가 남습니다.

`sorted(items, key=len)`은 표시용 목록을 새로 만듭니다. `key`에 넘긴 함수는 항목 하나를 받아 비교할 기준값을 돌려줘야 합니다. 여기서는 `len("키")`가 1이므로 두 글자인 “우산”보다 먼저 옵니다. `key=len(items)`라고 쓰면 정렬 기준을 물건마다 계산해야 할 자리에 정수가 들어가 TypeError가 납니다.

In [1]:
items = ["텀블러", "키", "우산", "장갑"]
by_length = sorted(items, key=len)  # len을 넘겨 각 이름의 길이를 비교합니다.
print(by_length)

['키', '우산', '장갑', '텀블러']


### key로 넘긴 함수에 입력 하나 넣어 보기

In [ ]:
word = "장갑"
print(len(word))
print(sorted(["장갑", "키"], key=len))

2
['키', '장갑']


2는 “장갑”의 비교값입니다. 정렬 결과에는 비교에 쓴 2와 1 대신 원래 물건 이름들이 들어 있습니다.

‘키’가 가장 짧아 맨 앞에 왔습니다. ‘우산’과 ‘장갑’은 각각 두 글자입니다. 비교 값이 같으면 sorted는 두 물건이 접수된 순서를 유지합니다. key=len()을 쓰면 단어를 한 개도 주지 않은 채 len부터 실행해 TypeError가 납니다.

### assert로 정렬 뒤 원본 검사하기

테스트에는 실행 뒤에도 지켜야 할 조건을 적습니다. assert 뒤의 식이 참이면 다음 줄로 넘어가고, 거짓이면 그 자리에서 AssertionError가 납니다. 아래에서 원래 접수 순서를 그대로 적은 까닭을 생각해 보세요.

In [2]:
# 출력용 목록을 만든 뒤에도 접수 순서를 다시 읽을 수 있어야 합니다.
assert items == ["텀블러", "키", "우산", "장갑"]
print(items)

['텀블러', '키', '우산', '장갑']


원본에서는 여전히 ‘텀블러’가 첫 물건입니다. items.sort(key=len)로 원본을 직접 정렬하면 이 테스트에서 차이가 드러납니다. 정렬 결과만 검사하면 놓치기 쉬운 변경입니다.

### 실습 1 · 같은 입력으로 두 목록 만들기

사전순과 길이순으로 번갈아 조회할 수 있게 arrange와 이름 자체를 반환하는 함수 by_name을 작성하세요. 두 번 모두 같은 items를 건네고 서로 다른 결과와 원본 순서를 assert로 검사하세요. 새 입력에서는 길이가 같은 ‘우산’과 ‘장갑’의 접수 순서를 바꿔 길이순 결과를 예측하세요.

In [3]:
items = ["텀블러", "키", "우산", "장갑"]
# by_name(item)과 arrange(names, key)를 작성하세요. 같은 items에 len과 by_name을 각각 넘기세요.
# 두 반환 목록과 원래 순서를 assert로 확인하세요. 길이가 같은 두 항목의 순서도 살펴보세요.

<details><summary>모범 답안과 실행 결과 보기</summary>
<pre><code>def by_name(item):
    return item

def arrange(names, key):
    return sorted(names, key=key)  # 호출자가 고른 함수를 정렬에 전달합니다.

short_first = arrange(items, len)
name_first = arrange(items, by_name)
assert short_first == [&quot;키&quot;, &quot;우산&quot;, &quot;장갑&quot;, &quot;텀블러&quot;]
assert name_first == [&quot;우산&quot;, &quot;장갑&quot;, &quot;키&quot;, &quot;텀블러&quot;]
assert items == [&quot;텀블러&quot;, &quot;키&quot;, &quot;우산&quot;, &quot;장갑&quot;]
print(short_first)
print(name_first)

changed = [&quot;텀블러&quot;, &quot;키&quot;, &quot;장갑&quot;, &quot;우산&quot;]
print(arrange(changed, len))</code></pre>
<p>실행 결과</p>
<pre>[&#x27;키&#x27;, &#x27;우산&#x27;, &#x27;장갑&#x27;, &#x27;텀블러&#x27;]
[&#x27;우산&#x27;, &#x27;장갑&#x27;, &#x27;키&#x27;, &#x27;텀블러&#x27;]
[&#x27;키&#x27;, &#x27;장갑&#x27;, &#x27;우산&#x27;, &#x27;텀블러&#x27;]</pre>
<p>이름순으로 볼 때에는 각 문자열 자체가 비교 값입니다. 새 접수 순서로 길이순 목록을 만들면 길이가 같은 ‘장갑’이 ‘우산’보다 먼저 나옵니다.</p>
</details>

### 기준 함수는 언제 실행될까요?

번호표를 붙일 때는 접수 순서대로, 작은 상자에 담을 때는 이름이 짧은 순서대로 물건을 봅니다. 정렬 함수에 기준을 넘긴다는 것은, 물건 하나를 받으면 **비교할 값 하나를 돌려주는 함수**를 넘긴다는 뜻입니다. 그 함수에 지금 당장 물건을 넣어 실행한 값과 함수 자체를 구별하세요.

In [4]:
def name_length(item):
    return len(item)  # 한 물건에서 비교에 사용할 숫자를 계산합니다.

print(name_length("장갑"))
print(sorted(["장갑", "키"], key=name_length))  # 물건을 고르는 일은 sorted가 합니다.

2
['키', '장갑']


첫 출력은 2입니다. 둘째 호출은 장갑과 키의 길이를 비교해 키를 앞으로 놓습니다. key=name_length("장갑")으로 바꾸면 key에 함수가 와야 할 자리에 숫자 2가 들어갑니다. 동료에게 “입력은 어떤 값이고 답으로 어떤 값을 돌려주나?”를 설명할 수 있으면 새 기준도 만들 수 있습니다.

### 전이 실습 · 접수 순번을 보존한 보관함 목록

접수표 한 건은 (보관함, 물건 이름)입니다. 먼저 접수 순서대로 원본을 출력하고, 다음에는 보관함 사전순으로 보되 같은 보관함 안에서는 접수 순서를 유지하세요. 길이 기준도 같은 조회 함수로 바꿔 써 보세요. 보관함을 구분하는 함수에 튜플 전체가 들어온다는 점에 주목하세요.

In [5]:
receipts = [("B", "장갑"), ("A", "우산"), ("A", "키")]
# 한 건을 받아 보관함을 돌려주는 함수와 별도의 길이 기준을 만드세요.

<details><summary>모범 풀이와 실행 결과 보기</summary>
<pre><code>receipts = [(&quot;B&quot;, &quot;장갑&quot;), (&quot;A&quot;, &quot;우산&quot;), (&quot;A&quot;, &quot;키&quot;)]
def by_box(receipt):
    return receipt[0]

def by_item_length(receipt):
    return len(receipt[1])

def view_receipts(records, criterion):
    return sorted(records, key=criterion)  # 사본을 정렬해 원본 접수 순서를 보존합니다.

print(view_receipts(receipts, by_box))
print(view_receipts(receipts, by_item_length))
print(receipts)</code></pre>
<pre>[(&#x27;A&#x27;, &#x27;우산&#x27;), (&#x27;A&#x27;, &#x27;키&#x27;), (&#x27;B&#x27;, &#x27;장갑&#x27;)]
[(&#x27;A&#x27;, &#x27;키&#x27;), (&#x27;B&#x27;, &#x27;장갑&#x27;), (&#x27;A&#x27;, &#x27;우산&#x27;)]
[(&#x27;B&#x27;, &#x27;장갑&#x27;), (&#x27;A&#x27;, &#x27;우산&#x27;), (&#x27;A&#x27;, &#x27;키&#x27;)]</pre>
<p>보관함 A 안의 우산과 키는 접수 순서대로 남습니다. 정렬 값이 같은 항목의 기존 순서를 지키는 성질을 안정 정렬이라고 합니다. 새 기준을 넣어도 조회 함수의 구현은 그대로입니다. 같은 보관함의 두 행 위치를 맞바꾸면 출력도 두 행만 맞바뀝니다.</p></details>

### 정렬한 접수 목록 보관하기

동점인 이름의 접수 순서가 뒤의 게시판까지 유지되는지 보려면 1장의 결과를 먼저 남겨야 합니다. 원래 목록에서 장갑과 우산의 자리를 바꿔 시험하세요.

In [ ]:
sorted_items = sorted(items, key=len)
save_work("b01.json", sorted_items)
print(sorted_items)

['키', '우산', '장갑', '텀블러']


기본 입력에서는 키가 첫 번째이고, 길이가 같은 우산과 장갑은 접수 순서대로입니다. 다음 장에서는 이 순서에 우선 물건 규칙을 더합니다.

## 2장 · 우선 목록을 기억하는 Strategy

Strategy(전략)는 호출하는 쪽에서 처리 규칙을 골라 넘기는 방식입니다. 여기서는 정렬 함수의 `key`에 다른 함수를 넣으면 순서가 바뀝니다. 우선순위 목록까지 기억해야 할 때는 호출 가능한 객체에 목록을 저장합니다.

객체를 만들면 `__init__`에서 받은 목록을 self.names에 보관합니다. `__call__`을 정의하면 그 객체에 함수처럼 괄호를 붙여 부를 수 있습니다. 아래 규칙은 우선 물건에는 0, 나머지에는 1을 줍니다. 여러 우선 물건 사이의 순서는 접수 순서를 따릅니다.

하나의 우선순위 규칙이 오늘의 물건 명단을 기억해야 합니다. `Priority(names)`로 객체를 만들 때 명단을 저장하고, 정렬할 때는 `__call__(self, item)`이 항목 한 개의 우선순위를 돌려줍니다. 아래에서는 객체를 정렬 기준으로 넘기기 전에 그 객체에 물건 한 개부터 넣어 봅니다.

### 접수 목록을 정렬 규칙에 넘기기

1장에서 저장한 순서를 먼저 읽습니다. 오늘은 비가 와서 우산을 찾는 사람이 많습니다. 우산만 맨 앞으로 보내고 다른 물건끼리의 순서는 유지해 보세요.

In [ ]:
ordered = read_work("b01.json", ["키", "우산", "장갑", "텀블러"])
print(ordered)

['키', '우산', '장갑', '텀블러']


첫 칸을 기록해 두세요. 다음 예제의 Priority는 우선 목록에 든 물건에 0을 돌려줍니다.

In [6]:
class Priority:
    def __init__(self, names):
        self.names = names

    def __call__(self, item):
        if item in self.names:  # 오늘 먼저 보여 줄 물건인지 확인합니다.
            return 0
        return 1

today = Priority(["우산"])
print(today("우산"))

0


0은 ‘우산’의 접수 번호가 아닙니다. 정렬에서 앞쪽에 둘 묶음이라는 뜻입니다. 이제 이 객체를 그대로 key에 넘길 수 있습니다.

In [7]:
print(sorted(items, key=today))  # sorted가 물건마다 today(물건)를 호출합니다.

['우산', '텀블러', '키', '장갑']


‘우산’이 앞으로 옮겨졌고 나머지는 원래 차례를 지켰습니다. 반대로 조건을 곧장 True나 False로 반환하면 어떤 결과가 나올까요? <mark>오름차순에서 False가 True보다 먼저 오므로 우선 물건이 뒤로 갑니다.</mark>

### 실습 · 두 항목의 우선순위가 같으면 어느 쪽이 앞설까요?

“키”와 “우산”을 모두 우선하는 규칙으로 아래 목록을 정렬하세요. 우선 목록에 적은 순서와 입력 순서가 다릅니다. 어느 쪽이 출력 순서를 결정하는지 확인하세요.

In [ ]:
items_to_sort = ["컵", "우산", "키"]
# 앞에서 만든 Priority 객체를 key에 넘겨 정렬 결과를 출력하세요.
# 같은 비교값을 받은 두 항목이 items_to_sort에서 어느 순서였는지 대조하세요.

<details><summary>정답 코드와 실행 결과</summary><pre><code>items_to_sort = [&quot;컵&quot;, &quot;우산&quot;, &quot;키&quot;]
priority = Priority([&quot;키&quot;, &quot;우산&quot;])
print(sorted(items_to_sort, key=priority))</code></pre><pre>[&#x27;우산&#x27;, &#x27;키&#x27;, &#x27;컵&#x27;]</pre><p>두 항목의 비교값은 모두 0입니다. 파이썬 정렬은 비교값이 같은 항목의 입력 순서를 유지하므로 우산이 먼저 나옵니다.</p></details>

### 실습 2 · 두 데스크의 우선 목록 분리하기

비 오는 날 데스크는 ‘우산’, 분실 신고 데스크는 ‘장갑’과 ‘키’를 우선합니다. 같은 items를 두 데스크에 넘기세요. 우선 목록을 적은 순서와 접수 순서가 다를 때 어느 것이 먼저인지 테스트를 쓰세요. desk[:2]처럼 대괄호 안에 :2를 쓰면 리스트의 앞 두 항목을 꺼낼 수 있습니다. 우선 목록이 비어 있으면 첫 접수 내역을 그대로 볼 수 있어야 합니다.

In [8]:
rain_rule = Priority(["우산"])
# 나머지 데스크의 기준을 만들고 같은 items를 두 번 정렬하세요.
# 우선 목록이 비었을 때도 원본과 비교하세요.

<details><summary>모범 답안과 실행 결과 보기</summary>
<pre><code>rain_rule = Priority([&quot;우산&quot;])
desk_rule = Priority([&quot;장갑&quot;, &quot;키&quot;])
rain = sorted(items, key=rain_rule)
desk = sorted(items, key=desk_rule)
assert rain[0] == &quot;우산&quot;
assert desk[:2] == [&quot;키&quot;, &quot;장갑&quot;]  # 같은 점수에서는 먼저 접수한 ‘키’가 앞섭니다.
assert sorted(items, key=Priority([])) == items
print(rain)
print(desk)</code></pre>
<p>실행 결과</p>
<pre>[&#x27;우산&#x27;, &#x27;텀블러&#x27;, &#x27;키&#x27;, &#x27;장갑&#x27;]
[&#x27;키&#x27;, &#x27;장갑&#x27;, &#x27;텀블러&#x27;, &#x27;우산&#x27;]</pre>
<p>두 객체에 다른 목록을 저장했지만 정렬 코드는 같습니다. 빈 목록을 주면 모든 이름의 점수가 1이라 접수 순서가 보존됩니다.</p>
</details>

### 우선순위 목록을 참조하거나 복사하면 무엇이 달라질까요?

아침에 확정한 목록은 그 시점의 규칙을 남겨야 하고, 실시간 접수 화면은 담당자가 고친 우선 표를 즉시 반영해야 합니다. 이름은 모두 Priority여도 두 요구는 다릅니다. 아래에서 상자에 새 물건을 받은 뒤 기존 규칙의 결과를 살펴보세요.

In [9]:
staff_choices = ["우산"]
live_rule = Priority(staff_choices)
staff_choices.append("키")  # 직원이 원래의 우선 목록을 수정했습니다.
print(live_rule("키"))

0


키가 0이 된 까닭은 두 이름이 같은 목록을 가리키기 때문입니다. 실시간 화면에서는 원하는 동작입니다. 오전 마감표처럼 규칙을 고정할 때는 만드는 순간의 목록을 복사해 넘길 수 있습니다. 어떤 방식이 맞는지는 입력 담당자와 출력 시점이 정해 줍니다.

### 정책 변경 실습 · 현재 안내와 마감표 한 장 만들기

접수 순서는 키, 지갑, 우산입니다. 아침에는 우산만 우선이었고 오후에 지갑을 우선 표 맨 뒤에 추가했습니다. 같은 접수 목록에서 ‘아침 확정’과 ‘현재’ 정렬 결과를 따로 보여 주세요. Priority가 우선 여부만 판단한다는 점을 반영해 두 우선 물건끼리의 순서도 예측하세요.

In [10]:
received_names = ["키", "지갑", "우산"]
priority_list = ["우산"]
# 우선 표를 고치기 전에 두 규칙을 만들고, 고친 뒤 같은 자료에 사용합니다.

<details><summary>모범 풀이와 실행 결과 보기</summary>
<pre><code>received_names = [&quot;키&quot;, &quot;지갑&quot;, &quot;우산&quot;]
priority_list = [&quot;우산&quot;]
morning_rule = Priority(priority_list.copy())  # 확정할 때 규칙을 복사합니다.
current_rule = Priority(priority_list)
priority_list.append(&quot;지갑&quot;)
print(sorted(received_names, key=morning_rule))
print(sorted(received_names, key=current_rule))</code></pre>
<pre>[&#x27;우산&#x27;, &#x27;키&#x27;, &#x27;지갑&#x27;]
[&#x27;지갑&#x27;, &#x27;우산&#x27;, &#x27;키&#x27;]</pre>
<p>아침 결과는 우산이 맨 앞입니다. 현재 결과에서는 지갑과 우산 모두 우선값 0이므로 원래 접수 순서에서 먼저 있던 지갑이 먼저 옵니다. ‘우선 표에 적은 순서대로’라는 요구라면 현재 기준이 주는 0과 1만으로는 부족합니다. 어떤 규칙을 구현했는지 결과로 확인해야 합니다.</p></details>

### Priority로 오늘의 게시 순서 정하기

우산을 먼저 찾도록 목록을 다시 정렬해 저장하세요. 우선 물건을 장갑으로 바꾸고 첫 칸만 달라지는지, 나머지의 접수 순서도 유지되는지 확인합니다.

In [ ]:
posting_order = sorted(ordered, key=Priority(["우산"]))
save_work("b02.json", posting_order)
print(posting_order)

['우산', '키', '장갑', '텀블러']


기본 결과의 첫 물건은 우산입니다. 기존 순서에서 우산을 꺼내 맨 앞으로 보낸 결과와 비교해 보세요.

### 실습 · 교대할 때 우선 물건 바꾸기

다음 담당자가 지정한 우선 물건을 Strategy에 넘겨 게시 순서를 저장하세요. 우산을 지정한 뒤 장갑으로 바꿔 3장의 첫 게시물이 달라지는지 확인합니다.

In [ ]:
priority_items = ["우산"]
# ordered를 Priority(priority_items)로 정렬하고 b02.json에 저장하세요.

<details><summary>코드와 기본 입력의 결과 펼치기</summary><pre><code>priority_items = [&quot;우산&quot;]
order = sorted(ordered, key=Priority(priority_items))
save_work(&quot;b02.json&quot;, order)
print(order[0])</code></pre>
<pre>우산</pre><p>우선 목록에 없는 물건들은 이전의 상대 순서를 유지합니다.</p></details>

## 3장 · Factory로 출력 형식 고르기

Factory(팩토리)는 요청한 종류에 맞는 객체를 골라 만들어 주는 함수나 객체입니다. 여기서는 문자열을 그대로 표시할지 굵게 표시할지에 따라 다른 포매터를 받습니다. 어느 쪽이든 `render("우산")`으로 결과를 읽습니다.

### 게시할 물건을 순서대로 읽기

2장의 게시 순서를 엽니다. 담당자가 첫 물건을 굵은 글씨로 표시할 수 있게 출력 형식을 골라 보세요.

이 과제는 게시할 물건이 하나 이상 있을 때 실행합니다. 1장에서 빈 접수 목록을 저장했다면 물건 한 건을 추가하고 1~2장의 저장 셀을 다시 실행하세요.

In [ ]:
posting_order = read_work("b02.json", ["우산", "키", "장갑", "텀블러"])
print(posting_order[0])

우산


우산이 나오면 다음 예제에 보낼 물건 이름을 확보했습니다. 다른 물건을 우선 처리했다면 그 이름으로 이어갑니다.

In [11]:
class Plain:
    def render(self, item):
        return item

class Bold:
    def render(self, item):
        return "**" + item + "**"  # 게시판이 굵게 표시하도록 원문에 별표를 남깁니다.

print(Bold().render("우산"))

**우산**


노트북에서 print로 읽으면 별표가 보입니다. 게시판에 보낼 원문에 별표가 들어갔는지를 이 상태에서 검사할 수 있습니다.

### make_formatter(kind)로 출력 객체 고르기

`make_formatter("bold")`는 굵은 글씨를 만드는 객체를 돌려줍니다. 글자를 실제로 바꾸는 호출은 그 객체의 `render("우산")`입니다. 지원하지 않는 이름이면 팩토리가 `ValueError`를 일으킵니다. 아래의 `publish`에서 객체를 고르는 줄과 글을 바꾸는 줄을 짚어 보세요.

`make_formatter(kind)`는 요청 형식에 맞는 객체를 골라 주는 팩토리 함수입니다. 반환된 객체의 `render(item)`을 호출해야 실제 출력 문자열이 생깁니다. 새 형식을 추가할 때 어느 함수가 형식 이름을 알아야 하고, 어느 함수는 `render`만 알면 되는지 두 호출을 나눠 읽으세요.

In [12]:
def make_formatter(kind):
    if kind == "plain":
        return Plain()
    if kind == "bold":
        return Bold()
    raise ValueError("지원하지 않는 형식: " + kind)

def publish(factory, kind, item):
    formatter = factory(kind)  # 생성은 팩터리에 맡기고 받은 객체에 출력을 요청합니다.
    return formatter.render(item)

print(publish(make_formatter, "plain", "우산"))

우산


plain을 받은 팩터리가 Plain 객체를 만들었습니다. publish에서는 어떤 객체를 받든 render(item) 한 번만 호출합니다.

### 실습 3 · 항목 기호 형식 추가하기

게시판 담당자가 ‘- 우산’처럼 줄 앞에 기호를 붙여 달라고 합니다. Bullet 포매터와 세 형식을 만드는 함수를 쓰세요. publish 본문을 그대로 두고 세 형식의 실제 문자열을 각각 기대값과 비교한 뒤 실습지에 기록하세요. 미등록 pdf를 요청하고 예외가 난 위치를 설명하세요. try에는 실행할 코드, except ValueError에는 그 예외가 났을 때 실행할 코드를 둡니다.

In [13]:
# Bullet.render는 물건 이름 앞에 '- '를 붙여 돌려줍니다.
# make_three는 기존 두 형식을 make_formatter에 맡길 수 있습니다.
# publish로 세 형식과 미등록 pdf를 각각 확인하세요.

<details><summary>모범 답안과 실행 결과 보기</summary>
<pre><code>class Bullet:
    def render(self, item):
        return &quot;- &quot; + item

def make_three(kind):
    if kind == &quot;bullet&quot;:
        return Bullet()
    return make_formatter(kind)  # 기존 두 형식과 잘못된 형식의 처리를 함께 물려받습니다.

plain = publish(make_three, &quot;plain&quot;, &quot;우산&quot;)
bold = publish(make_three, &quot;bold&quot;, &quot;우산&quot;)
bullet = publish(make_three, &quot;bullet&quot;, &quot;우산&quot;)
assert plain == &quot;우산&quot;
assert bold == &quot;**우산**&quot;
assert bullet == &quot;- 우산&quot;
print(plain)
print(bold)
print(bullet)
try:
    publish(make_three, &quot;pdf&quot;, &quot;우산&quot;)
except ValueError as error:
    print(error)</code></pre>
<p>실행 결과</p>
<pre>우산
**우산**
- 우산
지원하지 않는 형식: pdf</pre>
<p>pdf는 make_three에서 기존 팩터리로 전달됩니다. make_formatter 마지막 줄에서 ValueError가 나므로 publish의 render 줄에는 도달하지 않습니다. pdf를 plain처럼 조용히 처리하면 직원은 잘못 입력한 형식을 알아차리기 어렵습니다.</p>
</details>

### 형식 선택을 쉬운 예제로 따로 시험하기

게시판의 게시 위치가 틀렸는지 글자를 만드는 규칙이 틀렸는지를 구분하고 싶습니다. 먼저 실제 게시판 대신 “어떤 형식을 받았는지 알려 주는 객체”를 넘깁니다. publish가 어떤 순서로 일을 맡기는지 관찰하면, 새 형식을 추가할 때 수정할 부분이 드러납니다.

In [14]:
class Preview:
    def render(self, item):
        return "미리보기 " + item

def preview_factory(kind):
    print("요청 형식", kind)  # publish가 실제로 넘긴 이름을 확인합니다.
    return Preview()

print(publish(preview_factory, "bold", "카드"))

요청 형식 bold
미리보기 카드


첫 줄은 팩터리가 형식 이름을 받았다는 증거입니다. 둘째 줄에는 미리보기 포매터가 물건 이름으로 만든 문자열이 보입니다. 이 작은 대체 객체로는 실제 bold의 별표가 맞는지 알 수 없습니다. 팩터리의 전달을 확인한 뒤 실제 렌더링도 따로 검사해야 합니다.

### 납품 실습 · 대여 데스크의 이름표 추가

label 형식에는 ‘보관함 / 카드’처럼 보관 위치 안내를 붙입니다. plain과 bold를 쓰는 기존 경로와 미등록 pdf가 실패하는 동작을 보존하세요. 같은 publish로 세 입력을 걸어 보세요. 어느 입력이 형식 선택 단계에서 멈추는지 표시합니다.

In [15]:
item_to_publish = "카드"
formats_to_try = ["label", "plain", "pdf"]
# Label.render와 label_factory를 작성하세요. 같은 물건을 세 형식으로 게시해 보세요.
# pdf에서 발생한 ValueError와 앞선 두 문자열을 따로 확인하세요.

<details><summary>모범 풀이와 실행 결과 보기</summary>
<pre><code>class Label:
    def render(self, item):
        return &quot;보관함 / &quot; + item

def label_factory(kind):
    if kind == &quot;label&quot;:
        return Label()
    return make_formatter(kind)  # 이미 지원하는 형식과 오류 처리를 위임합니다.

print(publish(label_factory, &quot;label&quot;, &quot;카드&quot;))
print(publish(label_factory, &quot;plain&quot;, &quot;카드&quot;))
try:
    publish(label_factory, &quot;pdf&quot;, &quot;카드&quot;)
except ValueError as error:
    print(error)</code></pre>
<pre>보관함 / 카드
카드
지원하지 않는 형식: pdf</pre>
<p>label은 새 객체가, plain은 기존 Plain이 처리했습니다. pdf는 포매터를 만들기 전에 make_formatter에서 거절됩니다. 미등록 형식까지 plain으로 돌려버리면 담당자가 오타를 발견할 수 없습니다. 새 형식의 성공 사례와 미등록 형식 한 건을 함께 검수하세요.</p></details>

### Factory로 첫 줄을 굵게 만들어 보관하기

앞에서 정의한 팩터리에 형식 이름을 넘겨 게시판 첫 줄을 작성하세요. 같은 이름에 plain을 적용해 별표가 어디서 붙는지 비교합니다.

In [ ]:
first_item = posting_order[0]
line = publish(make_formatter, "bold", first_item)
save_work("b03.json", {"name": first_item, "line": line})
print(line)

**우산**


기본 예에서는 `**우산**`이라는 문자열입니다. 두 쌍의 별표는 팩터리가 골라 준 Bold 객체의 `render`에서 붙었습니다.

### 실습 · 요청한 형식으로 첫 게시물 작성하기

첫 물건 이름을 유지한 채 출력만 꾸미세요. bold 대신 plain으로 바꾸고 4장 앞부분에서 원문과 표시 문자열이 어떻게 달라졌는지 확인합니다.

In [ ]:
kind = "bold"
# 첫 물건에 publish를 호출하고 name, line을 b03.json에 저장하세요.

<details><summary>코드와 기본 입력의 결과 펼치기</summary><pre><code>kind = &quot;bold&quot;
item = posting_order[0]
line = publish(make_formatter, kind, item)
save_work(&quot;b03.json&quot;, {&quot;name&quot;: item, &quot;line&quot;: line})
print(line)</code></pre>
<pre>**우산**</pre><p>plain이면 별표가 빠져도 게시한 물건 이름은 우산입니다.</p></details>

## 4장 · Adapter와 Decorator로 기존 게시판 연결하기

Adapter(어댑터)는 기존 객체의 호출 형식을 다른 코드가 기대하는 모양에 맞춥니다. 예를 들어 새 코드가 `render("우산")`을 부르면 어댑터가 그 안에서 기존 객체의 `draw(label="우산")`을 호출합니다. Decorator(데코레이터)는 같은 호출 형식을 유지하면서 처리 전후에 기록 같은 동작을 더합니다.

### 두 게시판의 같은 물건 확인하기

3장의 게시 문구와 원래 물건 이름을 한 쌍으로 엽니다. 이제 이전 시스템에도 같은 물건을 표시하고 호출 기록을 남깁니다.

In [ ]:
post = read_work("b03.json", {"name": "우산", "line": "**우산**"})
print(post["name"], post["line"])

우산 **우산**


왼쪽은 어댑터에 넣을 이름이고 오른쪽은 이전 출력입니다. 별표까지 이름에 넣으면 이전 시스템에 별표가 남습니다.

In [16]:
class OldBoard:
    def draw(self, *, label):
        return "[" + label + "]"

print(OldBoard().draw(label="장갑"))  # 기존 게시판이 받는 방식입니다.

[장갑]


기존 게시판은 ‘[장갑]’을 만들어 줍니다. 하지만 publish는 render(item)을 부릅니다. OldBoard에 render가 없으니 그대로 연결하면 AttributeError가 납니다. Adapter(어댑터)는 바깥에서 받은 호출을 안쪽 객체가 이해하는 호출로 바꿉니다.

기존 게시판의 `draw(*, label)`에서 별표 뒤의 인자는 이름을 써서 넘겨야 합니다. 새 코드가 기대하는 호출은 `render(item)`입니다. 어댑터 안에서만 `item`을 `label=item`으로 넘기면, 게시판이나 호출자를 고치지 않고 두 인터페이스를 연결할 수 있습니다.

In [17]:
class BoardAdapter:
    def __init__(self, board):
        self.board = board  # 이미 만들어진 기존 게시판을 보관합니다.

    def render(self, item):
        return self.board.draw(label=item)

board = BoardAdapter(OldBoard())
print(board.render("장갑"))

[장갑]


출력의 대괄호는 OldBoard가 붙였습니다. 어댑터는 호출 모양만 바꿉니다. 이렇게 객체를 속성으로 보관해 일을 나누는 방법을 객체 합성이라고 합니다.

### Decorator로 render 결과를 유지하면서 호출 기록하기

`Logged(board, history)`는 기존 포매터와 기록용 리스트를 받습니다. `tracked.render("장갑")`을 부르면 먼저 안쪽 포매터가 문자열을 만듭니다. 이 호출이 성공하면 `history.append(item)`으로 원래 이름을 기록하고 안쪽의 반환값을 그대로 돌려줍니다.

In [18]:
class Logged:
    def __init__(self, inner, log):
        self.inner = inner
        self.log = log

    def render(self, item):
        line = self.inner.render(item)
        self.log.append(item)  # 출력에 성공한 호출만 이 목록에 남습니다.
        return line

history = []
tracked = Logged(board, history)
print(tracked.render("장갑"))
print(history)

[장갑]
['장갑']


게시할 한 줄은 여전히 ‘[장갑]’입니다. history에는 ‘장갑’이 하나 생겼습니다. 다음 호출도 같은 기록에 남기려면 그 호출에서도 같은 리스트를 써야 합니다.

### 감싸는 순서와 기록에 남는 이름

입력에 붙은 여백까지 표시되는 일이 생겼습니다. 아래 포매터는 양 끝의 공백을 지운 이름을 안쪽 포매터에 건넵니다. strip()은 새 문자열을 돌려줍니다. 두 포매터를 겹쳐 쓰면 맨 바깥 객체가 원래 입력을 가장 먼저 받습니다.

In [19]:
class Trimmed:
    def __init__(self, inner):
        self.inner = inner

    def render(self, item):
        return self.inner.render(item.strip())

print(Trimmed(board).render(" 장갑 "))

[장갑]


대괄호 안에 공백이 없습니다. 기록을 남기는 위치가 공백을 지우기 전인지 후인지는 화면 한 줄로는 알 수 없습니다.

### 실습 4 · 입력 원문과 정리한 이름 중 하나 기록하기

기존 게시판 앞에 어댑터를 놓고 Logged와 Trimmed의 순서를 바꿔 두 번 조립하세요. ‘ 장갑 ’을 각각 한 번 출력해 화면은 같은지, 각 로그에 어떤 문자열이 남는지 확인하세요. 한 조합만 다시 호출하고, 별도로 만든 Plain에는 기록을 붙이지 마세요. 호출한 조합의 기록만 하나 늘어야 합니다.

In [20]:
raw_log = []
clean_log = []
# 어댑터에 Logged, Trimmed를 두 순서로 붙이세요.
# 결과 문자열과 두 로그를 비교하고 한쪽만 다시 호출하세요.

<details><summary>모범 답안과 실행 결과 보기</summary>
<pre><code>raw_log = []
clean_log = []
adapter = BoardAdapter(OldBoard())
raw = Logged(Trimmed(adapter), raw_log)    # 바깥의 Logged가 공백까지 받습니다.
clean = Trimmed(Logged(adapter, clean_log))

assert raw.render(&quot; 장갑 &quot;) == clean.render(&quot; 장갑 &quot;) == &quot;[장갑]&quot;
print(raw_log, clean_log)
clean.render(&quot; 키 &quot;)
Plain().render(&quot;우산&quot;)
assert raw_log == [&quot; 장갑 &quot;]
assert clean_log == [&quot;장갑&quot;, &quot;키&quot;]
print(clean_log)</code></pre>
<p>실행 결과</p>
<pre>[&#x27; 장갑 &#x27;] [&#x27;장갑&#x27;]
[&#x27;장갑&#x27;, &#x27;키&#x27;]</pre>
<p>두 번째 조합에서는 Trimmed가 먼저 ‘장갑’을 만들고 그 이름이 로그에 남습니다. 같은 화면을 만들었어도 원문을 보존할지를 결정하는 곳은 두 객체를 조립한 줄입니다.</p>
</details>

### 연결은 성공했는데 실패 기록이 하나 남는다면

게시판은 빈 물건 이름을 받으면 게시를 거절하기로 했습니다. 성공 기록을 남기는 담당자에게는 세 번 시도해 두 번 게시했을 때 두 건만 보여야 합니다. 여기서는 게시 거절을 ValueError로 전달합니다. Logged가 안쪽 호출 **전과 후 중 언제** 이름을 저장하는지 확인하세요.

In [21]:
class AcceptedOnly:
    def render(self, item):
        if not item:
            raise ValueError("물건 이름이 비었습니다.")
        return "[" + item + "]"

published_items = []
checked = Logged(AcceptedOnly(), published_items)
try:
    checked.render("")
except ValueError as error:
    print(error)
print(published_items)  # 거절된 시도가 성공 기록에 섞였는지 확인합니다.

물건 이름이 비었습니다.
[]


안쪽 render에서 예외가 생기면 그다음 줄인 log.append까지 도달하지 않습니다. 실패한 시도도 보고해야 한다면 성공 기록과 시도 기록을 어디에 둘지 구분해 설계를 바꿔야 합니다. 예외를 빈 글자 하나로 바꾸면 바깥에서는 실패와 성공을 알아볼 수 없습니다.

### 통합 실습 · 공백뿐인 이름과 성공 한 건 구분하기

담당자가 공백 세 칸을 보내거나 ‘ 키 ’를 보냅니다. 양 끝 공백을 제거한 이름이 빈 문자열이면 거절하고, 성공한 경우에만 정리된 이름을 한 번 기록하세요. 이미 만든 Trimmed, Logged, AcceptedOnly를 조합해 해결합니다. 공백을 없애기 전에 검사했을 때 생길 문제도 설명하세요.

In [22]:
submitted_names = ["   ", " 키 "]
ok_names = []
# Trimmed, Logged, AcceptedOnly를 연결해 두 입력을 차례로 보내세요.
# 어느 호출에서 예외가 났고 처리 뒤 ok_names에 무엇이 남았나요?

<details><summary>모범 풀이와 실행 결과 보기</summary>
<pre><code>ok_names = []
board_input = Trimmed(Logged(AcceptedOnly(), ok_names))
for submitted in [&quot;   &quot;, &quot; 키 &quot;]:
    try:
        print(board_input.render(submitted))
    except ValueError as error:
        print(error)
print(ok_names)</code></pre>
<pre>물건 이름이 비었습니다.
[키]
[&#x27;키&#x27;]</pre>
<p>가장 바깥의 Trimmed가 입력을 고친 뒤 안쪽으로 건넵니다. 공백 세 칸은 빈 이름이어서 예외가 나고, 키는 게시된 뒤에만 기록됩니다. 공백 제거가 없으면 공백 세 칸도 비어 있지 않은 문자열이므로 검사에서 통과합니다.</p></details>

### Adapter와 Logged로 게시한 결과 기록하기

옛 게시판을 어댑터로 감싼 뒤 성공한 호출을 Logged로 기록하세요. 3장에서 이름이 바뀌었다면 이 게시판에도 같은 물건이 올라가는지 봅니다.

In [ ]:
published = []
desk = Logged(BoardAdapter(OldBoard()), published)
legacy_line = desk.render(post["name"])
save_work("b04.json", {"name": post["name"], "line": legacy_line, "log": published})
print(legacy_line, published)

[우산] ['우산']


기본 결과는 [우산]과 이름 하나가 든 목록입니다. 대괄호는 OldBoard가, 기록 한 건은 Logged가 만들었습니다.

### 실습 · 출력과 기록을 한 쌍으로 보관하기

새 기록 목록과 옛 게시판을 연결하고 앞에서 고른 물건을 한 번만 게시하세요. 같은 데스크에 두 번 요청하면 5장의 변경 전 기준에는 호출이 몇 건 남나요?

In [ ]:
trial_log = []
# Logged와 BoardAdapter로 감싸고 출력과 기록을 b04.json에 저장하세요.

<details><summary>코드와 기본 입력의 결과 펼치기</summary><pre><code>trial_log = []
desk = Logged(BoardAdapter(OldBoard()), trial_log)
line = desk.render(post[&quot;name&quot;])
save_work(&quot;b04.json&quot;, {&quot;name&quot;: post[&quot;name&quot;], &quot;line&quot;: line, &quot;log&quot;: trial_log})
print(len(trial_log))</code></pre>
<pre>1</pre><p>호출을 한 번 더 한 뒤 저장하면 출력이 같아도 기록은 두 건입니다. 두 건을 요구하는 서비스라면 뒤의 검사가 실패하는 이유를 설명할 수 있어야 합니다.</p></details>

## 5장 · 새 요구사항을 같은 테스트로 검증하기

새 형식을 추가해도 기존 호출의 결과가 유지되는지 같은 입력으로 검사합니다. 예를 들어 `tag` 형식에 “우산”을 보내면 “#우산”이 나와야 합니다. 호출 기록을 켜고 끄는 경우도 같은 테스트에서 비교합니다.

5장부터 새 커널에서 시작하려면 이 노트북 맨 앞의 파일 준비 셀을 실행하세요. 3장의 Plain·Bold와 make_formatter·publish, 4장의 OldBoard·BoardAdapter·Logged·Trimmed를 정의하는 예제 셀을 순서대로 실행하면 됩니다. 4장 파일이 없을 때는 아래 첫 셀이 우산의 게시 결과를 제공합니다.

### 기존 분기문에서 수정할 위치 찾기

아래 코드는 지금 운영 중인 세 형식을 한 함수에서 고릅니다. 세 형식 중 하나와 맞으면 return에서 함수가 끝납니다. 어느 조건에도 맞지 않으면 마지막 줄에서 예외가 납니다. tag를 추가하려면 예외를 일으키기 전에 새 형식의 출력을 정해야 합니다.

### 기존 게시판의 기준값 열기

5장에서는 4장의 게시 문자열과 기록을 변경 전 기준으로 삼습니다. 기존 입력과 기대 문자열을 먼저 확인하세요.

In [ ]:
old = read_work("b04.json", {"name": "우산", "line": "[우산]", "log": ["우산"]})
print(old["name"], old["line"])

우산 [우산]


표시 문자열의 괄호까지 일치해야 기존 게시판과 같은 동작입니다. 기록이 두 번 생기는 오류는 별도로 점검합니다.

In [23]:
def format_current(kind, item):
    if kind == "plain":
        return item
    if kind == "bold":
        return "**" + item + "**"
    if kind == "bullet":
        return "- " + item
    raise ValueError("지원하지 않는 형식: " + kind)

print(format_current("bullet", "키"))

- 키


‘- 키’는 지금 지원해야 하는 결과입니다. 새로운 형식이 잘 될 때도 기존 형식의 약속은 그대로 지켜야 합니다.

### 실습 3에서 넘겨받는 세 형식

마지막 문제만 다시 풀어도 되도록 3일째 완성할 팩터리를 아래에 제공합니다. 이 셀의 시작 동작은 분기문 버전과 같습니다. 앞서 직접 작성했다면 같은 입력으로 결과를 대조하세요.

In [24]:
class Bullet:
    def render(self, item):
        return "- " + item

def make_three(kind):
    if kind == "bullet":
        return Bullet()
    return make_formatter(kind)  # plain과 bold는 기존 팩터리가 만듭니다.

print(publish(make_three, "bullet", "키"))

- 키


앞에서 확인한 ‘- 키’가 그대로 나왔습니다. 새 tag 요청을 받으면 다른 형식은 make_three에 맡길 수 있습니다.

### check_board에 함수를 넘겨 두 구현 검사하기

`check_board(render)`는 검사할 함수 자체를 인자로 받습니다. 안에서 `render("tag", "우산", False, log)`를 실행해 반환값과 기록 목록을 따로 검사합니다. 기록을 꺼 둔 이 호출에서는 “#우산”이 반환되고 `log`는 빈 리스트여야 합니다.

`check_board(render)`에는 검사할 함수를 넘깁니다. 괄호를 붙여 미리 실행하면 검사가 시작되기 전에 함수의 결과를 넘기게 됩니다. 아래에서 각 `assert`의 왼쪽은 실제 호출 결과이고 오른쪽은 요구한 표시 형식입니다. 실패한 줄에 쓰인 입력으로 그 함수부터 다시 실행해 보세요.

In [25]:
def check_board(render):
    log = []  # 구현을 시험할 때마다 새 기록으로 시작합니다.
    assert render("plain", "우산", False, log) == "우산"
    assert render("bold", "우산", False, log) == "**우산**"
    assert render("bullet", "우산", False, log) == "- 우산"
    assert render("tag", "우산", False, log) == "#우산"
    assert log == []
    assert render("tag", "키", True, log) == "#키"
    assert log == ["키"]
    return log

방금 셀은 아직 검사할 함수를 건네받지 않았습니다. check_board에 구현을 넘겨 실행하면 네 형식의 출력과 기록을 차례로 비교합니다. assert가 멈춘 줄에서 기대한 값을 확인하세요.

### 실습 5 · 형식과 기록을 추가하고 유지할 구조 고르기

format_current를 활용한 분기문 버전과 3일째 팩터리 및 Logged를 활용한 조립 버전을 만드세요. 두 함수에 check_board를 똑같이 적용하세요. 미등록 형식에 기록을 켠 경우에도 요청이 실패하고 기록은 빈지 시험하세요. 이후 ‘다음 달에도 형식을 추가한다’는 조건에서 어느 쪽을 넘길지, 새 형식 또는 기록 정책을 바꿀 줄을 근거로 실습지에 적으세요.

In [26]:
# render_branch(kind, item, log_on, log)와 render_parts(...)를 작성하세요.
# check_board를 두 번 실행하고, 실패한 요청의 로그도 검사하세요.

<details><summary>모범 답안과 실행 결과 보기</summary>
<pre><code>def render_branch(kind, item, log_on, log):
    if kind == &quot;tag&quot;:
        line = &quot;#&quot; + item
    else:
        line = format_current(kind, item)
    if log_on:
        log.append(item)  # 형식 변환이 성공한 뒤에만 기록합니다.
    return line

class Tag:
    def render(self, item):
        return &quot;#&quot; + item

def make_final(kind):
    if kind == &quot;tag&quot;:
        return Tag()
    return make_three(kind)  # 앞에서 확인한 세 형식의 생성을 재사용합니다.

def render_parts(kind, item, log_on, log):
    formatter = make_final(kind)
    if log_on:
        formatter = Logged(formatter, log)
    return formatter.render(item)

print(check_board(render_branch))
print(check_board(render_parts))

for render in [render_branch, render_parts]:
    log = []
    try:
        render(&quot;pdf&quot;, &quot;키&quot;, True, log)
    except ValueError as error:
        print(error, log)
    else:
        raise AssertionError(&quot;미등록 형식을 허용했습니다.&quot;)
    assert log == []</code></pre>
<p>실행 결과</p>
<pre>[&#x27;키&#x27;]
[&#x27;키&#x27;]
지원하지 않는 형식: pdf []
지원하지 않는 형식: pdf []</pre>
<p>두 버전 모두 기록을 켠 ‘키’만 한 번 남깁니다. 미등록 pdf는 출력 전에 실패해 기록이 비어 있습니다. 형식이 자주 추가된다면 조립 버전을 선택할 수 있습니다. 새 포매터를 만들고 make_final에 등록해도 render_parts와 Logged는 그대로 둡니다. 네 형식이 고정된 작은 프로그램에서 한 함수만 읽고 싶다면 분기문 쪽을 택할 근거도 있습니다.</p>
</details>

### 짝에게 넘길 변경 기록

다른 물건 이름 하나와 기록 여부를 직접 정해 두 구현에 넣으세요. 실습지 마지막 칸에 입력, 기대값, 실제값을 한 줄로 적습니다. 다음 Git 수업에서 그 한 줄을 팀의 cases.md로 옮깁니다. 짝은 이 줄만 보고 자신의 노트북에서도 같은 결과가 나오는지 확인합니다.

### 교대 검수 · 어떤 변경을 코드 한 곳에서 끝낼 수 있을까

분기문으로 구현해도 같은 테스트를 통과할 수 있습니다. 오늘 추가된 요구는 ‘형식 하나 더’가 아니라 ‘한 데스크는 접수 이름의 원문을 기록하고, 다른 데스크는 다듬은 이름을 기록하자’입니다. 예시 입력 “ 키 ”와 “장갑”의 차이를 먼저 보세요. 두 데스크 모두 화면에는 같은 이름을 보여 주므로 화면만 비교해서는 실수를 찾을 수 없습니다.

### 동료 검수 · 출력은 같지만 운영 기록이 다른 두 데스크

두 데스크를 만들어 ‘ 키 ’ 한 건을 각각 게시하세요. 한쪽에는 ‘ 키 ’, 다른 쪽에는 ‘키’가 남아야 합니다. 이어서 원문 데스크에만 ‘장갑’을 보냈을 때 기록이 섞이지 않는지도 시험하세요. 마지막에 같은 두 요구를 if 문 하나로 구현한다면 어느 곳에 분기를 두어야 할지 짝에게 설명하세요.

In [27]:
first_history = []
second_history = []
# BoardAdapter(OldBoard())에 Logged와 Trimmed를 서로 다른 순서로 씌우세요.
# 같은 " 키 "를 두 창구에 보내서 출력 문자열과 각 기록의 공백을 비교하세요.

<details><summary>모범 풀이와 실행 결과 보기</summary>
<pre><code>first_history = []
second_history = []
legacy = BoardAdapter(OldBoard())
first_desk = Logged(Trimmed(legacy), first_history)  # 손질하기 전 원문을 기록합니다.
second_desk = Trimmed(Logged(legacy, second_history))
assert first_desk.render(&quot; 키 &quot;) == second_desk.render(&quot; 키 &quot;) == &quot;[키]&quot;
first_desk.render(&quot;장갑&quot;)
print(first_history)
print(second_history)</code></pre>
<pre>[&#x27; 키 &#x27;, &#x27;장갑&#x27;]
[&#x27;키&#x27;]</pre>
<p>기존 포매터와 어댑터를 바꾸지 않고 로그를 남기는 위치만 다르게 조립했습니다. 반대로 팀의 요구가 늘 plain 표기 한 가지이고 두 데스크가 같다면 두 객체를 따로 조립할 이유가 줄어듭니다. 팀에 넘길 때는 구조 이름보다 어느 수정에서 코드가 한 곳만 바뀌었는지 설명하세요.</p></details>

### 실습 · 구역별 안내 문구 유지하기

**센터장 요청**　“구역별로 접두 문구를 바꾸되 근무 교대 때 복원”

동문은 ‘동문 안내’, 서문은 ‘서문 안내’를 씁니다. 카드(동문), 키(서문), 우산(동문)을 차례로 게시하세요.

먼저 함수 인자로 접두 문구를 전달하고 다음에는 두 포매터 객체에 문구를 각각 보관하세요. 마지막 우산에도 동문 문구가 붙는지 확인한 뒤 새 구역을 추가해 보세요.

In [28]:
# Plain(), Bold()를 재사용합니다. 각자 접두 문구를 보관할 위치를 고르세요.
# 세 번의 결과 순서가 같고 기존 게시판의 표시 규칙이 유지되는지 확인합니다.

<details><summary>운영 사례의 모범 구현과 실행 결과 보기</summary><pre><code>def sign(prefix, formatter, item):
    return prefix + &quot; / &quot; + formatter.render(item)

class DeskSign:
    def __init__(self, prefix, formatter):
        self.prefix = prefix
        self.formatter = formatter

    def render(self, item):
        # 게시판 형식은 기존 포매터에 맡기고 데스크 안내만 덧붙입니다.
        return self.prefix + &quot; / &quot; + self.formatter.render(item)

east = DeskSign(&quot;동문 안내&quot;, Plain())
west = DeskSign(&quot;서문 안내&quot;, Bold())
using_objects = [east.render(&quot;카드&quot;), west.render(&quot;키&quot;), east.render(&quot;우산&quot;)]
using_inputs = [
    sign(&quot;동문 안내&quot;, Plain(), &quot;카드&quot;),
    sign(&quot;서문 안내&quot;, Bold(), &quot;키&quot;),
    sign(&quot;동문 안내&quot;, Plain(), &quot;우산&quot;),
]
expected = [&quot;동문 안내 / 카드&quot;, &quot;서문 안내 / **키**&quot;, &quot;동문 안내 / 우산&quot;]
assert using_inputs == expected
assert using_objects == expected  # 두 구현이 같은 실무 약속을 지키는지 봅니다.
print(using_objects)</code></pre><pre>[&#x27;동문 안내 / 카드&#x27;, &#x27;서문 안내 / **키**&#x27;, &#x27;동문 안내 / 우산&#x27;]</pre><p>서문을 쓴 다음에도 동문 객체에는 동문의 문구가 남습니다. 남문 추가는 다른 객체를 하나 더 만들거나 호출마다 남문 문구를 직접 넘기면 됩니다. 호출 때마다 담당자가 구역을 고르는 일회성 작업이면 함수가 읽기 쉽고, 한 데스크를 장시간 반복 사용할 때는 생성할 때 설정한 객체가 문구 전달의 반복을 줄입니다. 전체 센터가 공유하는 변경 가능한 전역 문구는 연속 호출에서 앞 설정이 덮이기 쉽습니다.</p></details>

### 같은 입력으로 새 조합을 검수해 Git에 넘기기

출력 전후 공백을 정리하는 Trimmed를 덧붙이세요. 지난 장이 납품한 문자열과 기록을 함께 대조하면 겉으로만 같은 동작을 가려낼 수 있습니다.

In [ ]:
new_log = []
replacement = Trimmed(Logged(BoardAdapter(OldBoard()), new_log))
incoming = " " + old["name"] + " "
line = replacement.render(incoming)
case = {"input": incoming, "expected": [old["line"], old["log"]], "observed": [line, new_log]}
save_work("git-case-b.json", case)
print(case["observed"] == case["expected"])

True


기본 예에서는 True입니다. `Trimmed`가 앞뒤 공백을 먼저 지웠으므로 화면에는 [우산]이 보이고 기록에는 우산 한 건만 남습니다. 두 객체를 감싸는 순서를 바꾸면 어떤 비교가 실패하는지 확인하세요.

### 실습 · 공백이 든 게시 요청으로 회귀 검사하기

기존 게시 결과와 기록을 한 번에 검사할 자료를 완성하세요. 다음에는 Logged와 Trimmed의 순서를 바꾸고, 겉으로는 같은 게시물인데 어느 항목이 실패하는지 6장의 동료에게 넘기세요.

In [ ]:
padded_name = " " + old["name"] + " "
# 새 로그를 쓰는 조합으로 padded_name을 처리하고 git-case-b.json에 저장하세요.

<details><summary>코드와 기본 입력의 결과 펼치기</summary><pre><code>padded_name = &quot; &quot; + old[&quot;name&quot;] + &quot; &quot;
trial_log = []
desk = Trimmed(Logged(BoardAdapter(OldBoard()), trial_log))
shown = desk.render(padded_name)
result = {&quot;input&quot;: padded_name, &quot;expected&quot;: [old[&quot;line&quot;], old[&quot;log&quot;]], &quot;observed&quot;: [shown, trial_log]}
save_work(&quot;git-case-b.json&quot;, result)
print(result[&quot;expected&quot;] == result[&quot;observed&quot;])</code></pre>
<pre>True</pre><p>두 객체를 반대 순서로 감싸면 출력은 같아도 로그에 공백이 남습니다. 이 실패를 기록한 채 동료가 같은 입력으로 재현하게 할 수 있습니다.</p></details>

### 다음 수업에 가져올 파일

지금 사용한 노트북 사본에 자기 입력과 실제 출력을 남기세요. [6~7장 팀 Git 활동지](../team-git/practice.md)에서 이 파일을 개인 예제로 제출합니다. 짝이 새 커널에서 처음부터 실행해 같은 결과를 낼 수 있도록 원장 버전과 입력 순서를 기록하세요.